* Autoregressive Model (like GPT) : predicts the next word based on the words that come before it
* Masked Language Model (like BERT) : hide random words and ask the model to fill in those blank


* XLNet consider all the words in a sentence as context

In [1]:
import pandas as pd
import numpy as np

!pip install cleantext
!pip install --upgrade clean-text


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00


In [2]:
from cleantext import clean
import re
from transformers import XLNetTokenizer, XLNetForSequenceClassification, Trainer, pipeline, TrainingArguments
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import datasets

import random

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
data_test = pd.read_csv('/content/drive/MyDrive/Mastering_Gen_AI_&_LLM_Engg_4_July_2026-11_Oct_2026/04_Intro_to_LLMs/emotion-labels-test.csv')
data_train = pd.read_csv('/content/drive/MyDrive/Mastering_Gen_AI_&_LLM_Engg_4_July_2026-11_Oct_2026/04_Intro_to_LLMs/emotion-labels-train.csv')
data_val = pd.read_csv('/content/drive/MyDrive/Mastering_Gen_AI_&_LLM_Engg_4_July_2026-11_Oct_2026/04_Intro_to_LLMs/emotion-labels-val.csv')

In [5]:
data_test.head()

,text,label
0,You must be knowing #blithe means (adj.) Happ...,joy
1,Old saying 'A #smile shared is one gained for ...,joy
2,Bridget Jones' Baby was bloody hilarious 😅 #Br...,joy
3,@Elaminova sparkling water makes your life spa...,joy
4,I'm tired of everybody telling me to chill out...,joy


In [6]:
len(data_train),len(data_test),len(data_val)

(3613, 3142, 347)

In [7]:
data = pd.concat([data_train,data_test,data_val],ignore_index=True)

In [8]:
data['text_clean'] = data['text'].apply(lambda x : clean(x,no_emoji=True))

In [9]:
data['text_clean']= data['text_clean'].apply(lambda x : re.sub('#@[^\s]+','',x))

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_977/3920821024.py:1: SyntaxWarning: invalid escape sequence '\s'
  data['text_clean']= data['text_clean'].apply(lambda x : re.sub('#@[^\s]+','',x))


In [10]:
data.head(20)

,text,label,text_clean
0,Just got back from seeing @GaryDelaney in Burs...,joy,just got back from seeing @garydelaney in burs...
1,Oh dear an evening of absolute hilarity I don'...,joy,oh dear an evening of absolute hilarity i don'...
2,Been waiting all week for this game ❤️❤️❤️ #ch...,joy,been waiting all week for this game #cheer #fr...
3,"@gardiner_love : Thank you so much, Gloria! Yo...",joy,"@gardiner_love : thank you so much, gloria! yo..."
4,I feel so blessed to work with the family that...,joy,i feel so blessed to work with the family that...
5,"Today I reached 1000 subscribers on YT!! , #go...",joy,"today i reached 1000 subscribers on yt!! , #go..."
6,"@Singaholic121 Good morning, love! Happy first...",joy,"@singaholic121 good morning, love! happy first..."
7,#BridgetJonesBaby is the best thing I've seen ...,joy,#bridgetjonesbaby is the best thing i've seen ...
8,Just got back from seeing @GaryDelaney in Burs...,joy,just got back from seeing @garydelaney in burs...
9,@IndyMN I thought the holidays could not get a...,joy,@indymn i thought the holidays could not get a...


In [11]:
data['label'].value_counts()

,count
label,
fear,2252
anger,1701
joy,1616
sadness,1533


In [12]:
data.groupby('label').size()

,0
label,
anger,1701
fear,2252
joy,1616
sadness,1533


In [13]:
g = data.groupby('label')
data = pd.DataFrame(g.apply(lambda x : x.sample(g.size().min()).reset_index(drop=True)))

/tmp/ipykernel_977/2412918944.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = pd.DataFrame(g.apply(lambda x : x.sample(g.size().min()).reset_index(drop=True)))


In [14]:
data

text    label  \
label                                                                      
anger   0     Dentist just said to me' I'm going to numb you...    anger   
        1     This all resent #India #Pakistan problem is sa...    anger   
        2     I'd pay good money to watch someone slap that ...    anger   
        3     And she got all angry telling me 'but what wou...    anger   
        4     I'm not use to getting up early asf this shit ...    anger   
...                                                         ...      ...   
sadness 1528  SOBRIETY®   trying to make being sober fashion...  sadness   
        1529               My life went from happy to unhappy..  sadness   
        1530  Bojack Horseman: the saddest show ever written...  sadness   
        1531  @reikimuse @MHChat I am beyond the sadness poi...  sadness   
        1532                        @rockcock64 cum and despair  sadness   

                                                     text_clean  
label                                                            
anger   0     dentist just said to me' i'm going to numb you...  
        1     this all resent #india #pakistan problem is sa...  
        2     i'd pay good money to watch someone slap that ...  
        3     and she got all angry telling me 'but what wou...  
        4     i'm not use to getting up early asf this shit ...  
...                                                         ...  
sadness 1528  sobriety trying to make being sober fashionabl...  
        1529               my life went from happy to unhappy..  
        1530  bojack horseman: the saddest show ever written...  
        1531  @reikimuse @mhchat i am beyond the sadness poi...  
        1532                        @rockcock64 cum and despair  

[6132 rows x 3 columns]

In [15]:
data['label'].value_counts()

,count
label,
anger,1533
fear,1533
joy,1533
sadness,1533


In [16]:
data['label_int'] = LabelEncoder().fit_transform(data['label'])

In [17]:
data['label_int'].unique()

array([0, 1, 2, 3])

In [18]:
NUM_LABELS = 4

In [19]:
train_split, test_split = train_test_split(data,train_size=0.8)
train_split, val_split = train_test_split(train_split,train_size=0.9)

In [20]:
len(train_split),len(test_split),len(val_split)

(4414, 1227, 491)

In [21]:
train_split

text    label  \
label                                                                      
anger   418   @JrDingy96 Ikr people still got a grudge again...    anger   
        539                           People fuckin irritate me    anger   
sadness 257   @TheRandomAnt @meg_m203 yeah, they figured tha...  sadness   
anger   823                 ICQ is just making me mad!!!😤 #icq     anger   
sadness 529   @TheBenNatan szn 3 &gt;&gt;&gt; szn 1 &gt;&gt;...  sadness   
...                                                         ...      ...   
        866   @nytimes media celebrated Don King endorsing #...  sadness   
        904    @xaninaany I'm so serious DM me I'll tell u more  sadness   
        740   Had to give up on The Blue Guitar, couldn't st...  sadness   
anger   198   very little frustrates me as much as misplacin...    anger   
fear    1051  @SadiqKhan #terrorism shouldn't be a way of li...     fear   

                                                     text_clean  label_int  
label                                                                       
anger   418   @jrdingy96 ikr people still got a grudge again...          0  
        539                           people fuckin irritate me          0  
sadness 257   @therandomant @meg_m203 yeah, they figured tha...          3  
anger   823                   icq is just making me mad!!! #icq          0  
sadness 529   @thebennatan szn 3 >>> szn 1 >>> szn 2. just t...          3  
...                                                         ...        ...  
        866   @nytimes media celebrated don king endorsing #...          3  
        904    @xaninaany i'm so serious dm me i'll tell u more          3  
        740   had to give up on the blue guitar, couldn't st...          3  
anger   198   very little frustrates me as much as misplacin...          0  
fear    1051  @sadiqkhan #terrorism shouldn't be a way of li...          1  

[4414 rows x 4 columns]

In [22]:
train_df = pd.DataFrame({
    'label' : train_split.label_int.values,
    'text': train_split.text_clean.values
})
train_df

,label,text
0,0,@jrdingy96 ikr people still got a grudge again...
1,0,people fuckin irritate me
2,3,"@therandomant @meg_m203 yeah, they figured tha..."
3,0,icq is just making me mad!!! #icq
4,3,@thebennatan szn 3 >>> szn 1 >>> szn 2. just t...
...,...,...
4409,3,@nytimes media celebrated don king endorsing #...
4410,3,@xaninaany i'm so serious dm me i'll tell u more
4411,3,"had to give up on the blue guitar, couldn't st..."
4412,0,very little frustrates me as much as misplacin...


In [23]:
test_df = pd.DataFrame({
    'label' : test_split.label_int.values,
    'text': test_split.text_clean.values
})
test_df

,label,text
0,2,"logically speaking, this can't be the worst i'..."
1,1,i stepped into the shower and my spidey senses...
2,0,@pbhushan1 @indianexpress so in your opinion i...
3,3,why the fuck does my mum want me to put corn i...
4,3,@simon_barrett i will watch @blairwitchmovie w...
...,...,...
1222,0,candice's pout is gonna take someone eye out m...
1223,3,hate to see y'all frown but i'd rather see him...
1224,3,i told my therapist that i'm on the dance team...
1225,3,"@ycfcnick the city is famous for the shambles,..."


In [24]:
train_df = datasets.Dataset.from_dict(train_df)
test_df = datasets.Dataset.from_dict(test_df)

In [25]:
dataset_dict = datasets.DatasetDict({
    'train':train_df,
    'test':test_df
})

In [26]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 4414
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 1227
    })
})

# Embeddings

In [27]:
tokenizer = XLNetTokenizer.from_pretrained('xlnet/xlnet-base-cased')

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

In [28]:
def tokenize_function(examples):
  return tokenizer(examples['text'],padding='max_length',max_length=128,truncation=True)

In [29]:
tokenized_datasets = dataset_dict.map(tokenize_function,batched=True)

Map:   0%|          | 0/4414 [00:00<?, ? examples/s]

Map:   0%|          | 0/1227 [00:00<?, ? examples/s]

In [30]:
tokenized_datasets['train']['text'][10]

"i stopped watching gotham cause they dropped off tabitha and barb's relationship and made her pine after the main"

In [31]:
tokenized_datasets['train']['input_ids'][10]

[5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 17,
 150,
 1773,
 2441,
 345,
 1714,
 1102,
 63,
 1695,
 177,
 8414,
 8972,
 101,
 21,
 17,
 19192,
 26,
 23,
 1498,
 21,
 140,
 62,
 12547,
 99,
 18,
 587,
 4,
 3]

In [32]:
tokenizer.decode(5)

'<pad>'

In [33]:
tokenized_datasets['train']['attention_mask'][10]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [34]:
small_train_dataset = tokenized_datasets['train'].shuffle(seed=42).select(range(100))
small_val_dataset = tokenized_datasets['test'].shuffle(seed=42).select(range(100))

# Fine Tune our Model

In [35]:
model = XLNetForSequenceClassification.from_pretrained('xlnet-base-cased',
                                                       num_labels = NUM_LABELS,
                                                       id2label = {
                                                           0 : 'anger',
                                                           1 : 'fear',
                                                           2 :'joy',
                                                           3 : 'sadness'
                                                       })

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  467MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

[transformers] XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.weight                  | UNEXPECTED | 
lm_loss.bias                    | UNEXPECTED | 
logits_proj.bias                | MISSING    | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 
sequence_summary.summary.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
!pip install evaluate


model.safetensors: reconstructing file:   0%|          |  0.00B /  467MB            

model.safetensors: downloading bytes:           |  0.00B            

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [37]:
import evaluate
metric = evaluate.load('accuracy')

In [38]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits,axis=1)
  return metric.compute(predictions=predictions, references=labels)

In [39]:
training_args =TrainingArguments(output_dir='test_trainer',
                  eval_strategy = 'epoch', num_train_epochs=3, report_to='none')

In [40]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = small_train_dataset ,
    eval_dataset = small_val_dataset,
    compute_metrics = compute_metrics
)

In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.416277,0.240000
2,No log,1.393697,0.220000
3,No log,1.410550,0.280000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39, training_loss=1.4246479914738581, metrics={'train_runtime': 23.2718, 'train_samples_per_second': 12.891, 'train_steps_per_second': 1.676, 'total_flos': 21366375321600.0, 'train_loss': 1.4246479914738581, 'epoch': 3.0})

In [42]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
No log,1.410550,3,0.280000


{'eval_loss': 1.4105502367019653, 'eval_accuracy': 0.28}

In [49]:
model.save_pretrained('/content/drive/MyDrive/Mastering_Gen_AI_&_LLM_Engg_4_July_2026-11_Oct_2026/04_Intro_to_LLMs/fine_tuned_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [51]:
fine_tuned_model = XLNetForSequenceClassification.from_pretrained("/content/drive/MyDrive/Mastering_Gen_AI_&_LLM_Engg_4_July_2026-11_Oct_2026/04_Intro_to_LLMs/fine_tuned_model")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

In [54]:
clf = pipeline("text-classification", fine_tuned_model, tokenizer = tokenizer)

In [64]:
rand_int = random.randint(0,len(val_split))
answer = clf(val_split['text_clean'][rand_int], top_k=None)
print(val_split['text_clean'][rand_int])
print(answer)

i cant live anymore my roblox got termianted :(((((((((((((((((((((((( #sad #killme #lol #robloxgamer
[{'label': 'anger', 'score': 0.27280041575431824}, {'label': 'joy', 'score': 0.2591171860694885}, {'label': 'fear', 'score': 0.24875031411647797}, {'label': 'sadness', 'score': 0.21933214366436005}]


/tmp/ipykernel_977/788812440.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  answer = clf(val_split['text_clean'][rand_int], top_k=None)
/tmp/ipykernel_977/788812440.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(val_split['text_clean'][rand_int])
